# Stage 2 — Subtask 5: Augmentation Strategy
**Leaf-JEPA IRP** | Dataset Preparation

In supervised learning, augmentation affects regularisation.
In SSL, augmentation **defines what invariances the model learns** — a fundamentally different role.

This notebook defines and justifies all three augmentation pipelines, exports the configuration, and demonstrates sample augmented images.

**Pipelines:**
1. **Pretraining** — aggressive, label-invariant, protects disease colour
2. **Fine-tuning** — moderate (standard) + label-mixing for 1–5% label regime
3. **Evaluation** — deterministic, identical across all methods

**Alternatives included:**
- CutMix + Mixup (1–5% label fractions only)
- RandAugment (documented as ablation alternative)


In [1]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path("..").resolve().parent
OUTPUTS_DIR  = PROJECT_ROOT / "stage2_dataset_preparation/outputs"
AUG_OUT      = OUTPUTS_DIR / "augmentation"
AUG_OUT.mkdir(parents=True, exist_ok=True)
print("Setup complete.")


Setup complete.


## 1. Disease Colour Protection Principle

The primary diagnostic signal in plant disease imagery is **colour deviation from healthy green**:
- Chlorosis (yellowing): G↓, R↑ vs healthy
- Necrosis (browning/blackening): all channels ↓
- Rust pustules: R↑↑, G↓

**Consequence:** Hue and saturation jitter must be kept minimal across all phases. Hue > 0.1 rotates disease colours into random positions in colour space, destroying disease identifiability.


In [2]:
PRETRAINING_AUG = {
    "phase": "pretraining",
    "pytorch_transforms": [
        {
            "name": "RandomResizedCrop",
            "params": {"size": 224, "scale": [0.2, 1.0], "ratio": [0.75, 1.33]},
            "justification": (
                "Multi-scale sampling: forces SSL model to understand disease at multiple "
                "magnifications. Scale 0.2–1.0 is standard for I-JEPA (Assran et al., 2023)."
            ),
        },
        {
            "name": "RandomHorizontalFlip",  "params": {"p": 0.5},
            "justification": "Disease patterns are orientation-invariant.",
        },
        {
            "name": "RandomVerticalFlip", "params": {"p": 0.5},
            "justification": "Leaf images are captured at arbitrary orientations in field.",
        },
        {
            "name": "RandomRotation", "params": {"degrees": 30},
            "justification": "Rotation invariance. Limited to ±30° to avoid excessive black padding.",
        },
        {
            "name": "ColorJitter",
            "params": {"brightness": 0.3, "contrast": 0.3, "saturation": 0.1, "hue": 0.05},
            "justification": (
                "Handles lighting variation. "
                "CONSERVATIVE saturation (0.1) and hue (0.05): disease colour IS the signal. "
                "Strong colour jitter teaches colour-invariance — opposite of what we need."
            ),
            "risk": "MEDIUM if saturation > 0.1 or hue > 0.05. Do not increase these bounds.",
        },
        {
            "name": "GaussianBlur", "params": {"kernel_size": [3, 9], "sigma": [0.1, 2.0]},
            "justification": "Handles camera focus variation in field images.",
        },
        {
            "name": "RandomGrayscale", "params": {"p": 0.05},
            "justification": "Forces minimal colour-independent feature learning. 5% only.",
            "risk": "DANGEROUS if increased to p > 0.2.",
        },
        {
            "name": "RandomErasing",
            "params": {"p": 0.3, "scale": [0.02, 0.15], "ratio": [0.3, 3.3], "value": 0},
            "justification": "Simulates partial occlusion (overlapping leaves in field).",
        },
    ],
    "explicitly_excluded": [
        "Strong hue shifting (hue > 0.2) — destroys disease colour signal",
        "Elastic deformation — may destroy lesion shape morphology (concentric rings in blight)",
        "CutMix/Mixup — invalid for SSL (no labels to mix)",
    ],
}

FINETUNING_AUG = {
    "phase": "fine-tuning",
    "standard_policy": {
        "label": ">=10% label fraction",
        "pytorch_transforms": [
            {"name": "RandomResizedCrop", "params": {"size": 224, "scale": [0.6, 1.0]},
             "justification": "Moderate scale. Retains disease context for correct classification."},
            {"name": "RandomHorizontalFlip", "params": {"p": 0.5}},
            {"name": "ColorJitter",
             "params": {"brightness": 0.2, "contrast": 0.2, "saturation": 0.05, "hue": 0.02},
             "justification": "Very conservative. Disease colour must be preserved for classification."},
        ],
        "excluded": [
            "Gaussian blur — may soften lesion boundaries needed for classification",
            "Random grayscale — colour IS the classification signal during fine-tuning",
        ],
    },
    "low_label_policy": {
        "label": "1%–5% label fraction",
        "rationale": (
            "At 1–5% of 54K images (~380–1,900 images across 38 classes), standard augmentation "
            "provides insufficient regularisation. CutMix and Mixup are the most effective "
            "label-mixing regularisers for this extreme low-data regime."
        ),
        "additional_transforms": [
            {
                "name": "CutMix", "params": {"alpha": 1.0, "p": 0.5},
                "justification": "Patches crops across images; mixes labels proportionally. Applied at batch level.",
                "implementation": "Apply in training loop, not in Dataset. Use soft cross-entropy loss.",
            },
            {
                "name": "Mixup", "params": {"alpha": 0.2, "p": 0.3},
                "justification": "Linear interpolation of image pairs and labels. Complementary to CutMix.",
            },
        ],
        "caveat": "CutMix/Mixup may slightly reduce performance at >=25% labels. Use standard policy there.",
    },
    "randaugment_alternative": {
        "label": "RandAugment (ablation comparison)",
        "params": {"N": 2, "M": 9},
        "rationale": (
            "Policy-searched augmentation. Run as ablation vs manual policy. "
            "Expected: manual policy better because it explicitly excludes colour-destroying ops."
        ),
        "constraint": "MUST exclude: Solarize, Equalize, Posterize, AutoContrast",
        "citation": "Cubuk et al. (2020). RandAugment. CVPR.",
    },
}

EVALUATION_AUG = {
    "phase": "evaluation",
    "critical_rule": "DETERMINISTIC. IDENTICAL across ALL methods. No random transforms.",
    "pytorch_transforms": [
        {"name": "Resize", "params": {"size": 256},
         "justification": "Standardise scale before crop."},
        {"name": "CenterCrop", "params": {"size": 224},
         "justification": "Deterministic crop. Centre typically contains leaf subject."},
        {"name": "ToTensor", "params": {}},
        {"name": "Normalize",
         "params": {"mean": "from outputs/preprocessing/normalisation_stats.json",
                    "std": "from outputs/preprocessing/normalisation_stats.json"},
         "justification": "Normalise by training set statistics."},
    ],
    "explicitly_excluded": [
        "ANY random transform",
        "Test-Time Augmentation (TTA) in main results — use only as labelled ablation",
    ],
}

aug_config = {"pretraining": PRETRAINING_AUG, "finetuning": FINETUNING_AUG, "evaluation": EVALUATION_AUG}
with open(AUG_OUT / "augmentation_config.json", "w") as f:
    json.dump(aug_config, f, indent=2)
print(f"Augmentation config saved.")
print(f"Pretraining transforms : {len(PRETRAINING_AUG['pytorch_transforms'])}")
print(f"Fine-tuning (standard) : {len(FINETUNING_AUG['standard_policy']['pytorch_transforms'])}")
print(f"Fine-tuning (low-label): +{len(FINETUNING_AUG['low_label_policy']['additional_transforms'])} label-mixing")
print(f"Evaluation transforms  : {len(EVALUATION_AUG['pytorch_transforms'])}")


Augmentation config saved.
Pretraining transforms : 8
Fine-tuning (standard) : 3
Fine-tuning (low-label): +2 label-mixing
Evaluation transforms  : 4


## 2. PyTorch Transform Code (importable)

This cell outputs ready-to-use torchvision transform code for all training scripts.


In [3]:
# ── Load normalisation stats ──────────────────────────────────────────────────
norm_path = Path("../outputs/preprocessing/normalisation_stats.json")
if norm_path.exists():
    ns = json.loads(norm_path.read_text())
    NORM_MEAN, NORM_STD = ns["mean"], ns["std"]
    print(f"Using dataset-specific stats: mean={NORM_MEAN}, std={NORM_STD}")
else:
    NORM_MEAN = [0.485, 0.456, 0.406]  # Placeholder — update after running ST4
    NORM_STD  = [0.229, 0.224, 0.225]
    print("⚠️  Using ImageNet placeholder stats. Run ST4 (04_preprocessing.ipynb) first.")

# ── Transform definitions (copy into training scripts) ───────────────────────
transform_code = f'''
import torchvision.transforms as T

NORM_MEAN = {NORM_MEAN}
NORM_STD  = {NORM_STD}

def get_pretrain_transform():
    return T.Compose([
        T.RandomResizedCrop(224, scale=(0.2, 1.0), ratio=(0.75, 1.33)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.RandomRotation(degrees=30),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1, hue=0.05),
        T.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0)),
        T.RandomGrayscale(p=0.05),
        T.ToTensor(),
        T.Normalize(mean=NORM_MEAN, std=NORM_STD),
        T.RandomErasing(p=0.3, scale=(0.02, 0.15), ratio=(0.3, 3.3), value=0),
    ])

def get_finetune_transform(low_label: bool = False):
    base = T.Compose([
        T.RandomResizedCrop(224, scale=(0.6, 1.0)),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.05, hue=0.02),
        T.ToTensor(),
        T.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ])
    return base  # CutMix/Mixup applied at batch level in training loop for low_label=True

def get_eval_transform():
    return T.Compose([
        T.Resize(256),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ])
'''
print(transform_code)

# Save as importable .py
(AUG_OUT / "transforms.py").write_text(transform_code)
print(f"\n✅ Saved to outputs/augmentation/transforms.py  (copy into your training scripts)")
print("✅ SUBTASK 5 COMPLETE")


Using dataset-specific stats: mean=[0.465809, 0.487659, 0.409572], std=[0.19489, 0.169946, 0.213739]

import torchvision.transforms as T

NORM_MEAN = [0.465809, 0.487659, 0.409572]
NORM_STD  = [0.19489, 0.169946, 0.213739]

def get_pretrain_transform():
    return T.Compose([
        T.RandomResizedCrop(224, scale=(0.2, 1.0), ratio=(0.75, 1.33)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.RandomRotation(degrees=30),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1, hue=0.05),
        T.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0)),
        T.RandomGrayscale(p=0.05),
        T.ToTensor(),
        T.Normalize(mean=NORM_MEAN, std=NORM_STD),
        T.RandomErasing(p=0.3, scale=(0.02, 0.15), ratio=(0.3, 3.3), value=0),
    ])

def get_finetune_transform(low_label: bool = False):
    base = T.Compose([
        T.RandomResizedCrop(224, scale=(0.6, 1.0)),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.2, c